# 04 · Functional Toolkits: `functools` & `itertools` 🆕

**Module 09 – Advanced Functions** · Estimated time: 50 min

> ❗ Not covered in either source (apart from `reduce`). These two modules make loops shorter, faster and more expressive.

### 🎯 What you will learn
- `functools`: `partial`, `cache`/`lru_cache`, `reduce`, `singledispatch`, `wraps`, `total_ordering`, `cached_property`
- `itertools`: infinite iterators, slicing & filtering, grouping, combinatorics, 🆕 `pairwise` (3.10) and `batched` (3.12)

## Part A: `functools`

### `partial`: pre-fill some arguments

In [1]:
from functools import partial

def power(base, exponent):
    return base ** exponent

square = partial(power, exponent=2)
cube = partial(power, exponent=3)
print(square(5), cube(2))

int_from_binary = partial(int, base=2)
print(int_from_binary("1011"))

print_line = partial(print, sep=" | ", end=" ✓\n")
print_line("a", "b", "c")

25 8
11
a | b | c ✓


### `cache` and `lru_cache`: memoization

In [2]:
from functools import cache, lru_cache
import time

@lru_cache(maxsize=256)           # keeps the 256 most recently used results
def slow_square(n):
    time.sleep(0.05)
    return n * n

t = time.perf_counter(); slow_square(9); first = time.perf_counter() - t
t = time.perf_counter(); slow_square(9); second = time.perf_counter() - t
print(f"first call {first:.3f}s, cached call {second:.6f}s")
print(slow_square.cache_info())
slow_square.cache_clear()

first call 0.055s, cached call 0.000109s
CacheInfo(hits=1, misses=1, maxsize=256, currsize=1)


> ❗ Arguments must be **hashable** (no lists/dicts), and only cache **pure** functions (same input → same output, no side effects).

### `reduce` (recap from Module 05)

In [3]:
from functools import reduce
print(reduce(lambda acc, x: acc * x, [1, 2, 3, 4, 5], 1))

120


### `singledispatch`: one function, different implementations per type

In [4]:
from functools import singledispatch

@singledispatch
def describe(value):
    return f"Something: {value!r}"

@describe.register
def _(value: int):
    return f"An integer with {len(str(abs(value)))} digit(s)"

@describe.register
def _(value: list):
    return f"A list of {len(value)} items"

@describe.register
def _(value: str):
    return f"A string of length {len(value)}"

for v in [42, [1, 2, 3], "hello", 3.14]:
    print(describe(v))

An integer with 2 digit(s)
A list of 3 items
A string of length 5
Something: 3.14


Also in `functools` (covered elsewhere): `@wraps` (decorators), `@total_ordering` (comparisons), `@cached_property` (properties).

## Part B: `itertools`

All `itertools` functions return **lazy iterators**; wrap with `list()` to see the values.

### Infinite iterators

In [5]:
from itertools import count, cycle, repeat, islice

print(list(islice(count(10, 5), 4)))          # 10, 15, 20, 25, ...
print(list(islice(cycle("ABC"), 7)))          # A B C A B C A ...
print(list(repeat("hi", 3)))
for i, color in zip(range(5), cycle(["red", "green"])):
    print(i, color, end=" | ")

[10, 15, 20, 25]
['A', 'B', 'C', 'A', 'B', 'C', 'A']
['hi', 'hi', 'hi']
0 red | 1 green | 2 red | 3 green | 4 red | 

### Slicing & filtering

In [6]:
from itertools import takewhile, dropwhile, filterfalse, compress

nums = [1, 3, 5, 6, 7, 9, 2]
print(list(takewhile(lambda x: x % 2, nums)))     # take while odd → [1, 3, 5]
print(list(dropwhile(lambda x: x % 2, nums)))     # drop while odd → [6, 7, 9, 2]
print(list(filterfalse(lambda x: x % 2, nums)))   # the opposite of filter → [6, 2]
print(list(compress("ABCDEF", [1, 0, 1, 0, 1, 1])))
print(list(islice(range(100), 10, 20, 3)))        # slice ANY iterator

[1, 3, 5]
[6, 7, 9, 2]
[6, 2]
['A', 'C', 'E', 'F']
[10, 13, 16, 19]


### Combining & accumulating

In [7]:
from itertools import chain, accumulate, zip_longest, starmap
import operator

print(list(chain([1, 2], (3, 4), "ab")))
print(list(chain.from_iterable([[1, 2], [3], [4, 5]])))   # flatten one level
print(list(accumulate([1, 2, 3, 4, 5])))                    # running totals
print(list(accumulate([3, 1, 4, 1, 5], max)))               # running maximum
print(list(accumulate([1, 2, 3, 4], operator.mul)))         # running product
print(list(zip_longest("AB", "xyz", fillvalue="-")))
print(list(starmap(pow, [(2, 3), (3, 2), (10, 0)])))

[1, 2, 3, 4, 'a', 'b']
[1, 2, 3, 4, 5]
[1, 3, 6, 10, 15]
[3, 3, 4, 4, 5]
[1, 2, 6, 24]
[('A', 'x'), ('B', 'y'), ('-', 'z')]
[8, 9, 1]


### 🆕 `pairwise` (3.10) and `batched` (3.12)

In [8]:
from itertools import pairwise, batched

prices = [100, 104, 101, 110]
print(list(pairwise(prices)))
print([b - a for a, b in pairwise(prices)])          # day-to-day changes

print(list(batched(range(10), 3)))                   # chunks of 3
try:
    list(batched(range(10), 3, strict=True))          # 3.13+: require equal-sized batches
except ValueError as e:
    print("ValueError:", e)

[(100, 104), (104, 101), (101, 110)]
[4, -3, 9]
[(0, 1, 2), (3, 4, 5), (6, 7, 8), (9,)]
ValueError: batched(): incomplete batch


### `groupby`: group **consecutive** items by a key

In [9]:
from itertools import groupby

data = [("fruit", "apple"), ("fruit", "banana"), ("veg", "carrot"), ("fruit", "cherry")]
for key, group in groupby(data, key=lambda x: x[0]):
    print(key, [name for _, name in group])

print("--- sort first to group ALL items with the same key:")
for key, group in groupby(sorted(data), key=lambda x: x[0]):
    print(key, [name for _, name in group])

print([(k, len(list(g))) for k, g in groupby("aaabbcaa")])   # run-length encoding

fruit ['apple', 'banana']
veg ['carrot']
fruit ['cherry']
--- sort first to group ALL items with the same key:
fruit ['apple', 'banana', 'cherry']
veg ['carrot']
[('a', 3), ('b', 2), ('c', 1), ('a', 2)]


### Combinatorics

In [10]:
from itertools import product, permutations, combinations, combinations_with_replacement

print(list(product("AB", [1, 2])))                    # Cartesian product (nested loops)
print(list(product([0, 1], repeat=3)))                # all 3-bit patterns
print(list(permutations("ABC", 2)))                   # order matters
print(list(combinations("ABCD", 2)))                  # order doesn't matter
print(list(combinations_with_replacement("AB", 2)))
import math
print(math.perm(5, 2), math.comb(5, 2))               # just the counts

[('A', 1), ('A', 2), ('B', 1), ('B', 2)]
[(0, 0, 0), (0, 0, 1), (0, 1, 0), (0, 1, 1), (1, 0, 0), (1, 0, 1), (1, 1, 0), (1, 1, 1)]
[('A', 'B'), ('A', 'C'), ('B', 'A'), ('B', 'C'), ('C', 'A'), ('C', 'B')]
[('A', 'B'), ('A', 'C'), ('A', 'D'), ('B', 'C'), ('B', 'D'), ('C', 'D')]
[('A', 'A'), ('A', 'B'), ('B', 'B')]
20 10


### Example: find all pairs of items that fit a budget

In [11]:
menu = {"tea": 20, "coffee": 40, "sandwich": 90, "cake": 60}
budget = 100
pairs = [(a, b) for (a, pa), (b, pb) in combinations(menu.items(), 2) if pa + pb <= budget]
print(pairs)

[('tea', 'coffee'), ('tea', 'cake'), ('coffee', 'cake')]


### `tee`: split one iterator into several independent ones

In [12]:
from itertools import tee
a, b = tee(iter([1, 2, 3]))
print(list(a), list(b))

[1, 2, 3] [1, 2, 3]


## 📝 Summary

| Need | Tool |
|---|---|
| fix some args | `functools.partial` |
| memoize | `functools.cache` / `lru_cache` |
| type-based dispatch | `functools.singledispatch` |
| counters / repeating | `count`, `cycle`, `repeat` |
| slice/limit any iterator | `islice`, `takewhile`, `dropwhile` |
| flatten / join iterators | `chain`, `chain.from_iterable` |
| running totals | `accumulate` |
| neighbours / chunks | `pairwise`, `batched` |
| group consecutive | `groupby` (sort first!) |
| all combinations | `product`, `permutations`, `combinations` |

## 🏋️ Practice
1. Use `accumulate` to compute a running balance from transactions `[100, -20, -30, 50]`.
2. Use `groupby` to count consecutive identical characters in `"aaabccddd"` → `a3b1c2d3`.
3. Use `product` to generate all PIN codes of length 3 from digits `"012"`.
4. Use `combinations` to find all pairs of numbers in `[1, 4, 6, 9, 11]` that sum to 15.
5. Use `partial` to create `log_error = partial(print, "[ERROR]")`.

<details><summary>Show solutions</summary>

```python
from itertools import accumulate, groupby, product, combinations
from functools import partial

print(list(accumulate([100, -20, -30, 50])))
print("".join(f"{k}{len(list(g))}" for k, g in groupby("aaabccddd")))
print(["".join(p) for p in product("012", repeat=3)])
print([p for p in combinations([1, 4, 6, 9, 11], 2) if sum(p) == 15])
log_error = partial(print, "[ERROR]")
log_error("disk full")
```
</details>